# Testing Notebook for eval.py

In [2]:
import torch
import sys
import os
from munch import Munch

# Add src to path to import modules
sys.path.append('src')

from eval import (
    gen_standard,
    gen_scaled_query,
    gen_subspace,
    eval_batch,
    aggregate_metrics,
    eval_model
)
from samplers import get_data_sampler
from tasks import get_task_sampler

## Mock Objects for Testing

In [8]:

class MockModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.name = "mock_model"
        self.layer = torch.nn.Linear(20, 1)
        
    def forward(self, xs, ys, inds=None):
        return self.layer(xs).squeeze(-1)

class MockTask:
    def evaluate(self, xs):
        return torch.ones_like(xs[..., 0])
    
    def get_metric(self):
        return lambda pred, true: torch.mean((pred - true)**2, dim=-1)

def mock_task_sampler():
    return MockTask()

n_dims = 20
n_points = 10
b_size = 4

data_sampler = get_data_sampler('gaussian', n_dims)

## Testing Data Generation Functions

In [9]:
print("Testing gen_standard...")
xs, xs_p = gen_standard(data_sampler, n_points, b_size)
assert xs.shape == (b_size, n_points, n_dims)
assert xs_p is None
print("ok")

print("Testing gen_scaled_query...")
xs, xs_p = gen_scaled_query(data_sampler, n_points, b_size, scale=2.0)
assert xs.shape == (b_size, n_points, n_dims)
assert xs_p.shape == (b_size, n_points, n_dims)
assert torch.allclose(xs * 2.0, xs_p)
print("ok")

print("Testing gen_subspace...")
xs, xs_p = gen_subspace(data_sampler, n_points, b_size, frac=0.5)
assert xs.shape == (b_size, n_points, n_dims)
assert xs_p is None
print("ok")

Testing gen_standard...
ok
Testing gen_scaled_query...
ok
Testing gen_subspace...
ok


## Testing Evaluation Functions

In [10]:
print("Testing eval_batch...")
model = MockModel()
xs, _ = gen_standard(data_sampler, n_points, b_size)
metrics = eval_batch(model, mock_task_sampler, xs)
assert metrics.shape == (b_size, n_points)
print("ok")

Testing eval_batch...


AssertionError: 

## Testing Metric Aggregation

In [6]:
print("Testing aggregate_metrics...")
metrics_tensor = torch.randn(100, n_points)
results = aggregate_metrics(metrics_tensor)
assert 'mean' in results
assert 'std' in results
assert 'bootstrap_low' in results
assert 'bootstrap_high' in results
assert len(results['mean']) == n_points
print("ok")

Testing aggregate_metrics...
ok


## Integration Test: eval_model

In [7]:
print("Testing eval_model...")
model = MockModel()
results = eval_model(
    model,
    task_name='linear_regression',
    data_name='gaussian',
    n_dims=n_dims,
    n_points=n_points,
    prompting_strategy='standard',
    num_eval_examples=64,
    batch_size=b_size
)
assert 'mean' in results
assert len(results['mean']) == n_points
print("ok")

Testing eval_model...


RuntimeError: The size of tensor a (4) must match the size of tensor b (10) at non-singleton dimension 1